In [ ]:
from langchain_classic.vectorstores import Chroma
from langchain_core.documents import Document
import requests
from langchain_core.embeddings import Embeddings

In [ ]:
class ServerEmbedding(Embeddings):

    def embed_documents(self, texts):
        resp = requests.post(
            "http://172.17.11.82:8000/embed/dense",
            json={"texts": texts}
        )
        return resp.json()["embeddings"]

    def embed_query(self, text):
        resp = requests.post(
            "http://172.17.11.82:8000/embed/dense",
            json={"texts": [text]}
        )
        return resp.json()["embeddings"][0]

In [ ]:
embeddings = ServerEmbedding()

In [ ]:
texts = ["my name is Nima", "I am a data Manager", "I have a beatiful girl"]
docs = [Document(page_content=text) for text in texts]

In [ ]:
vectorstore = Chroma.from_documents(documents=docs,
                                    embedding=embeddings,
                                    persist_directory=f"E:\AI Projects\AI Course\VS") 

In [ ]:
vectorstore.get() 

In [ ]:
vectorstore.get(ids='ab18bde4-e86b-4311-b031-eb958dc2f22a')

In [ ]:
len(vectorstore.get()['documents'])

In [ ]:
added_document = Document(page_content = 'I am using AI to improve my ability',
                          metadata = {
                              'Course Title':'Introduction to AI',
                              'Lecture Title':'RAG'
                          })
vectorstore.add_documents([added_document])

In [ ]:
vectorstore.get('1e8702a2-76fc-484b-aa67-a79fff2f154d')

In [ ]:
vectorstore.delete(ids=['76d5f2c1-e138-45c2-80dd-eb4b81cac304',
  '503ea300-3f3b-4515-a3c7-6a355a2a07fe'])

In [ ]:
updated_document = Document(page_content = 'I am using AI to improve my ability',
                            metadata = {
                                'Course Title':'Introduction to AI',
                                'Lecture Title':'AI'
                          })

In [ ]:
retrieved_document = vectorstore.similarity_search(query="what is Nima's Job?", k=2)
print(retrieved_document)
for i in retrieved_document:
    print(i.page_content)


In [ ]:
retrieved_document_s = vectorstore.max_marginal_relevance_search(query="what is Nima's Job?",
                                                               k=3,
                                                               lambda_mult=0.7
                                                            #    filter={"Lecture Title":
                                                            #            "Programming language & software Employed in Data Science - ..."}
                                                                       ) #for better result



In [60]:
print(retrieved_document_s)
for i in retrieved_document_s:
    print(i.page_content)

[Document(metadata={}, page_content='my name is Nima'), Document(metadata={}, page_content='I am a data Manager'), Document(metadata={'Course Title': 'Introduction to AI', 'Lecture Title': 'RAG'}, page_content='I am using AI to improve my ability')]
my name is Nima
I am a data Manager
I am using AI to improve my ability


In [54]:
retriever = vectorstore.as_retriever(search_type = 'mmr',
                                     search_kwargs = {'k':3, 
                                                      'lambda_mult': 0.7})

In [55]:
question = "what is Nima's Job?"
retrieved_runnable = retriever.invoke(question)

In [56]:
retrieved_runnable

[Document(metadata={}, page_content='my name is Nima'),
 Document(metadata={}, page_content='I am a data Manager'),
 Document(metadata={'Course Title': 'Introduction to AI', 'Lecture Title': 'RAG'}, page_content='I am using AI to improve my ability')]